In [1]:
# 06_visualization.ipynb

# %% [markdown]
# # Regulatory-Constrained Fairness Framework for Credit Scoring
# ## 06. Visualization
#
# Publication-ready grayscale figures for the RCFS results and sensitivity analysis.
# - seaborn, grayscale palette, no titles/captions (captions handled in the manuscript)
# - every figure saved as both PNG and PDF at 600 dpi into results/figures
#
# Figures:
#   fig1_rcfs_by_technique   — RCFS per dataset x technique (grouped bars)
#   fig2_component_scores     — compliance component scores (c_perf/c_fair/c_expl) per dataset
#   fig3_gamma_curves         — RCFS vs gamma, one panel per dataset
#   fig4_bootstrap_ci         — RCFS with bootstrap 95% CI per dataset x technique

# %%
# Imports and global style
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style="white", context="paper")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "savefig.bbox": "tight",
    "figure.dpi": 150,
})

DPI = 600
DATASET_ORDER = ["German Credit", "Taiwan CC Default", "Credit Card Approval"]
TECH_ORDER = ["baseline", "reweighing", "reject_option"]
TECH_LABELS = {"baseline": "Baseline", "reweighing": "Reweighing (pre)",
               "reject_option": "Reject option (post)"}
# grayscale per technique
TECH_GRAY = {"baseline": "0.25", "reweighing": "0.55", "reject_option": "0.80"}

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=DPI)
    plt.close(fig)

# %%
# Load result tables produced by notebooks 04 and 05
rcfs_df = pd.read_csv(os.path.join(TAB_DIR, "04_rcfs_by_technique.csv"))
sweep_df = pd.read_csv(os.path.join(TAB_DIR, "04_rcfs_gamma_sweep.csv"))
boot = pd.read_csv(os.path.join(TAB_DIR, "05_sensitivity_bootstrap.csv"))

# %%
# Figure 1: RCFS per dataset x technique (grouped bars), grayscale
piv = rcfs_df.pivot(index="dataset", columns="technique", values="RCFS").reindex(
    index=DATASET_ORDER, columns=TECH_ORDER)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
x = np.arange(len(DATASET_ORDER))
width = 0.25
for i, tech in enumerate(TECH_ORDER):
    ax.bar(x + (i - 1) * width, piv[tech].values, width,
           label=TECH_LABELS[tech], color=TECH_GRAY[tech],
           edgecolor="black", linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels(DATASET_ORDER, rotation=15, ha="right")
ax.set_ylabel("RCFS")
ax.set_ylim(0, max(0.8, piv.values.max() * 1.15))
ax.legend(frameon=False, fontsize=9, loc="upper right")
sns.despine(ax=ax)
save_fig(fig, "fig1_rcfs_by_technique")

# %%
# Figure 2: compliance component scores per dataset (baseline technique), grouped bars.
# Shows why compliance is what it is: c_perf, c_fair, c_expl side by side.
base_rows = rcfs_df[rcfs_df["technique"] == "baseline"].set_index("dataset").reindex(DATASET_ORDER)
comp_cols = ["c_perf", "c_fair", "c_expl"]
comp_labels = ["Performance", "Fairness", "Explainability"]
comp_gray = ["0.20", "0.50", "0.78"]
comp_hatch = ["", "//", ".."]

fig, ax = plt.subplots(figsize=(6.4, 4.0))
x = np.arange(len(DATASET_ORDER))
width = 0.25
for i, (col, lab, g, h) in enumerate(zip(comp_cols, comp_labels, comp_gray, comp_hatch)):
    ax.bar(x + (i - 1) * width, base_rows[col].values, width,
           label=lab, color=g, edgecolor="black", linewidth=0.7, hatch=h)
ax.set_xticks(x)
ax.set_xticklabels(DATASET_ORDER, rotation=15, ha="right")
ax.set_ylabel("Component satisfaction score")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=9, loc="upper right")
sns.despine(ax=ax)
save_fig(fig, "fig2_component_scores")

# %%
# Figure 3: RCFS vs gamma, one panel per dataset (line per technique), grayscale
fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(10.5, 3.4), sharey=True)
tech_marker = {"baseline": "o", "reweighing": "s", "reject_option": "^"}
tech_ls = {"baseline": "-", "reweighing": "--", "reject_option": ":"}
for ax, ds in zip(axes, DATASET_ORDER):
    sub = sweep_df[sweep_df["dataset"] == ds]
    for tech in TECH_ORDER:
        s = sub[sub["technique"] == tech].sort_values("gamma")
        ax.plot(s["gamma"], s["RCFS"], marker=tech_marker[tech],
                linestyle=tech_ls[tech], color=TECH_GRAY[tech],
                markerfacecolor="white", markeredgecolor="black",
                linewidth=1.4, markersize=5, label=TECH_LABELS[tech])
    ax.set_title(ds, fontsize=10)
    ax.set_xlabel(r"$\gamma$")
    sns.despine(ax=ax)
axes[0].set_ylabel("RCFS")
axes[-1].legend(frameon=False, fontsize=8, loc="best")
fig.tight_layout()
save_fig(fig, "fig3_gamma_curves")

# %%
# Figure 4: RCFS with bootstrap 95% CI per dataset x technique, grayscale
fig, ax = plt.subplots(figsize=(7.0, 4.0))
x = np.arange(len(DATASET_ORDER))
width = 0.25
for i, tech in enumerate(TECH_ORDER):
    sub = boot[boot["technique"] == tech].set_index("dataset").reindex(DATASET_ORDER)
    means = sub["RCFS_mean"].values
    lo = means - sub["CI_low"].values
    hi = sub["CI_high"].values - means
    ax.bar(x + (i - 1) * width, means, width, label=TECH_LABELS[tech],
           color=TECH_GRAY[tech], edgecolor="black", linewidth=0.7,
           yerr=[lo, hi], capsize=3, error_kw={"elinewidth": 1.0, "ecolor": "black"})
ax.set_xticks(x)
ax.set_xticklabels(DATASET_ORDER, rotation=15, ha="right")
ax.set_ylabel("RCFS (95% bootstrap CI)")
ax.legend(frameon=False, fontsize=9, loc="upper right")
sns.despine(ax=ax)
save_fig(fig, "fig4_bootstrap_ci")

# %%
# List saved figures
saved = sorted(f for f in os.listdir(FIG_DIR)
               if f.startswith("fig") and f.endswith((".png", ".pdf")))
for f in saved:
    print(f)

fig1_rcfs_by_technique.pdf
fig1_rcfs_by_technique.png
fig2_component_scores.pdf
fig2_component_scores.png
fig3_gamma_curves.pdf
fig3_gamma_curves.png
fig4_bootstrap_ci.pdf
fig4_bootstrap_ci.png


In [3]:
# %%
# Figure 5: adversarial RCFS vs adv_lambda, one line per dataset (grayscale).
# Shows that the optimal adversarial strength differs across datasets.
adv_sweep = pd.read_csv(os.path.join(TAB_DIR, "07_adversarial_lambda_sweep.csv"))

ds_style = {
    "German Credit":        {"marker": "o", "ls": "-",  "gray": "0.15"},
    "Taiwan CC Default":    {"marker": "s", "ls": "--", "gray": "0.45"},
    "Credit Card Approval": {"marker": "^", "ls": ":",  "gray": "0.70"},
}

fig, ax = plt.subplots(figsize=(6.0, 4.0))
for ds in DATASET_ORDER:
    s = adv_sweep[adv_sweep["dataset"] == ds].sort_values("adv_lambda")
    st = ds_style[ds]
    ax.plot(s["adv_lambda"], s["RCFS"], marker=st["marker"], linestyle=st["ls"],
            color=st["gray"], markerfacecolor="white", markeredgecolor="black",
            linewidth=1.5, markersize=6, label=ds)
ax.set_xlabel(r"Adversarial strength $\lambda$")
ax.set_ylabel("RCFS")
ax.legend(frameon=False, fontsize=9)
sns.despine(ax=ax)
save_fig(fig, "fig5_adversarial_lambda")

# %%
# Figure 6: fairness vs performance trade-off along the lambda sweep (grayscale).
# c_fair and c_perf on the same axes to show what the adversary trades off, per dataset.
fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(10.5, 3.4), sharey=True)
for ax, ds in zip(axes, DATASET_ORDER):
    s = adv_sweep[adv_sweep["dataset"] == ds].sort_values("adv_lambda")
    ax.plot(s["adv_lambda"], s["c_fair"], marker="o", linestyle="-",
            color="0.15", markerfacecolor="white", markeredgecolor="black",
            linewidth=1.4, markersize=5, label="Fairness")
    ax.plot(s["adv_lambda"], s["c_perf"], marker="s", linestyle="--",
            color="0.55", markerfacecolor="white", markeredgecolor="black",
            linewidth=1.4, markersize=5, label="Performance")
    ax.set_title(ds, fontsize=10)
    ax.set_xlabel(r"$\lambda$")
    sns.despine(ax=ax)
axes[0].set_ylabel("Component score")
axes[-1].legend(frameon=False, fontsize=8, loc="best")
fig.tight_layout()
save_fig(fig, "fig6_adversarial_tradeoff")

# %%
# Refresh the saved-figure listing
saved = sorted(f for f in os.listdir(FIG_DIR)
               if f.startswith("fig") and f.endswith((".png", ".pdf")))
for f in saved:
    print(f)

fig1_rcfs_by_technique.pdf
fig1_rcfs_by_technique.png
fig2_component_scores.pdf
fig2_component_scores.png
fig3_gamma_curves.pdf
fig3_gamma_curves.png
fig4_bootstrap_ci.pdf
fig4_bootstrap_ci.png
fig5_adversarial_lambda.pdf
fig5_adversarial_lambda.png
fig6_adversarial_tradeoff.pdf
fig6_adversarial_tradeoff.png
